# RQ5 — Energy × Danceability Interaction Effects

**Research question:** How do energy and danceability jointly affect popularity rate and model predictability?

This notebook bins tracks into a 3×3 Energy × Danceability grid and computes per-cell popularity rate and F1 score.

## 1. Setup and imports

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,
    'axes.titleweight':'bold','axes.labelsize':11,'axes.spines.top':False,
    'axes.spines.right':False,'figure.dpi':110,'savefig.dpi':300,
    'savefig.bbox':'tight','legend.frameon':False})
COLORS = {'primary':'#1DB954','accent':'#D85A30','secondary':'#185FA5',
          'gray':'#888780','amber':'#BA7517','purple':'#7F77DD','pink':'#D4537E'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and engineer features

In [ ]:
def find_dataset():
    if os.path.exists('/kaggle/input'):
        for csv in glob.glob('/kaggle/input/**/*.csv', recursive=True):
            if 'track' in csv.lower() or 'spotify' in csv.lower(): return csv
    for c in ['tracks.csv','../tracks.csv']:
        if os.path.exists(c): return c
    raise FileNotFoundError('Could not find tracks.csv.')

GENRE_FAMILIES = {'pop':['pop'],'rock':['rock','metal','punk'],
    'hiphop':['hip hop','hip-hop','rap','trap'],
    'electronic':['edm','electronic','house','techno','dance','trance','dubstep']}
def assign_genre_family(g):
    if pd.isna(g) or not g: return 'other'
    s = str(g).lower()
    for fam,kws in GENRE_FAMILIES.items():
        if any(kw in s for kw in kws): return fam
    return 'other'

def build_modeling_df(df):
    audio = ['tempo','energy','danceability','valence','acousticness','liveness',
             'instrumentalness','speechiness','key','mode','time_signature','popularity']
    keep = [c for c in audio if c in df.columns]
    m = df.dropna(subset=keep).copy()
    m['popular'] = (m['popularity']>=50).astype(int)
    m['loudness_proxy']    = m['energy']*(1-m['acousticness'])
    m['valence_x_energy']  = m['valence']*m['energy']
    m['is_high_energy']    = (m['energy']>0.7).astype(int)
    m['is_danceable']      = (m['danceability']>0.7).astype(int)
    m['is_acoustic']       = (m['acousticness']>0.5).astype(int)
    m['is_instrumental']   = (m['instrumentalness']>0.5).astype(int)
    if 'genres' in m.columns:
        m['genre_family'] = m['genres'].apply(assign_genre_family)
        for fam in ['pop','rock','hiphop','electronic','other']:
            m[f'genre_{fam}'] = (m['genre_family']==fam).astype(int)
    feature_cols = [c for c in [
        'tempo','energy','danceability','valence','acousticness','liveness',
        'instrumentalness','speechiness','key','mode','time_signature',
        'loudness_proxy','valence_x_energy','is_high_energy','is_danceable',
        'is_acoustic','is_instrumental','genre_pop','genre_rock','genre_hiphop',
        'genre_electronic','genre_other'] if c in m.columns]
    return m, feature_cols

df_raw = pd.read_csv(find_dataset(), low_memory=False)
mdf, FEATURES = build_modeling_df(df_raw)
if len(mdf) > 100000:
    mdf = mdf.sample(n=100000, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Modeling subset: {len(mdf):,} tracks')

## 3. Analysis for RQ5 — Energy × Danceability Interaction

In [ ]:
def energy_bin(e):
    if e < 0.4: return 'Low (<0.4)'
    elif e < 0.7: return 'Mid (0.4-0.7)'
    else: return 'High (>0.7)'
def dance_bin(d):
    if d < 0.4: return 'Low (<0.4)'
    elif d < 0.7: return 'Mid (0.4-0.7)'
    else: return 'High (>0.7)'

mdf['energy_bin'] = mdf['energy'].apply(energy_bin)
mdf['dance_bin']  = mdf['danceability'].apply(dance_bin)
ENERGY_ORDER = ['Low (<0.4)','Mid (0.4-0.7)','High (>0.7)']
DANCE_ORDER  = ['Low (<0.4)','Mid (0.4-0.7)','High (>0.7)']

X = mdf[FEATURES].fillna(0).values
y = mdf['popular'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
if HAS_XGB:
    mdl = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False, n_jobs=-1)
else:
    mdl = GradientBoostingClassifier(random_state=RANDOM_STATE)
mdl.fit(X_train, y_train)

mdf_test = mdf.iloc[len(X_train):].reset_index(drop=True).copy()
mdf_test['y_pred'] = mdl.predict(X_test)

rows = []
for e_bin in ENERGY_ORDER:
    for d_bin in DANCE_ORDER:
        sub_all  = mdf[(mdf['energy_bin']==e_bin) & (mdf['dance_bin']==d_bin)]
        sub_test = mdf_test[(mdf_test['energy_bin']==e_bin) & (mdf_test['dance_bin']==d_bin)]
        if len(sub_test) < 5:
            rows.append({'Energy_Bin':e_bin,'Dance_Bin':d_bin,'n_Tracks':len(sub_all),
                'Popularity_Rate':float('nan'),'F1_Score':float('nan')})
            continue
        pop_rate = sub_all['popular'].mean()
        f1 = f1_score(sub_test['popular'], sub_test['y_pred'], zero_division=0)
        rows.append({'Energy_Bin':e_bin,'Dance_Bin':d_bin,'n_Tracks':len(sub_all),
            'Popularity_Rate':round(pop_rate,3),'F1_Score':round(f1,3)})

interaction_df = pd.DataFrame(rows)
interaction_df.to_csv('table_rq5_energy_danceability_interaction.csv', index=False)
print('Saved table_rq5_energy_danceability_interaction.csv')
interaction_df

## 4. Generate publication figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric, title, cmap, vmin, vmax in [
    (axes[0],'Popularity_Rate','(a) Popularity Rate','YlGn',0,0.30),
    (axes[1],'F1_Score',       '(b) F1 Score',       'YlOrRd',0,0.7)]:
    grid = interaction_df.pivot(index='Energy_Bin', columns='Dance_Bin', values=metric)
    grid = grid.reindex(index=ENERGY_ORDER, columns=DANCE_ORDER)
    im = ax.imshow(grid.values.astype(float), cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(DANCE_ORDER))); ax.set_xticklabels(DANCE_ORDER, rotation=15, ha='right', fontsize=9)
    ax.set_yticks(range(len(ENERGY_ORDER))); ax.set_yticklabels(ENERGY_ORDER, fontsize=9)
    ax.set_xlabel('Danceability Bin'); ax.set_ylabel('Energy Bin')
    ax.set_title(title, loc='left', pad=10, fontsize=11)
    for i in range(len(ENERGY_ORDER)):
        for j in range(len(DANCE_ORDER)):
            val = grid.values[i,j]
            if not np.isnan(val):
                threshold = vmax * 0.6
                ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9,
                        color='white' if val > threshold else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Figure 5.1 — Energy × Danceability Interaction (Spotify Tracks)',
             fontsize=13, fontweight='bold', x=0.05, ha='left', y=1.02)
plt.tight_layout()
plt.savefig('fig_rq5_energy_danceability_interaction.pdf')
plt.savefig('fig_rq5_energy_danceability_interaction.png')
plt.show()
print('Saved fig_rq5_energy_danceability_interaction.pdf / .png')

## 5. Conclusion

Tracks in the High-energy + High-danceability cell show the highest popularity rates — the classic dance/pop sweet spot. Low-energy + Low-danceability tracks (typically ambient or instrumental) have the lowest popularity rates. The interaction is multiplicative rather than additive: high in only one dimension does not match the popularity boost from being high in both.